In [1]:
import os
import time
import torch
import torch.nn as nn
from models.FLB_ViT import FLBViT
from utils.dataset import get_cifar10_dataloaders

In [2]:
# Device and Hyperparameters
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 128
learning_rate = 3e-4
weight_decay = 1e-2
epochs = 20
max_steps = 12
threshold = 1e-2

# Initialize Model, Loss, and Optimizer
model = FLBViT(
    img_size=32,
    patch_size=4,
    in_channels=3,
    num_classes=10,
    embed_dim=192,
    depth=3,
    num_heads=6,
    dropout=0.1
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(), lr=learning_rate, weight_decay=weight_decay
)

# Inspect Parameter Counts
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
jepa_params = sum(
    p.numel() for name, p in model.named_parameters()
    if p.requires_grad and "jepa" in name
)
print(f"Device: {device}")
print(f"Total Trainable Parameters: {total_params:,}")
print(f"Inference Backbone Parameters (excluding JEPA): {total_params - jepa_params:,}")
print(f"JEPA Predictor Parameters: {jepa_params:,}")

Device: cuda
Total Trainable Parameters: 3,104,459
Inference Backbone Parameters (excluding JEPA): 2,328,587
JEPA Predictor Parameters: 775,872


In [3]:
# Sanity Check: Verify shapes and gradient flow on a single dummy batch
model.train()
optimizer.zero_grad()

dummy_images = torch.randn(4, 3, 32, 32, device=device)
dummy_labels = torch.randint(0, 10, (4,), device=device)

logits, ce_loss_val, aux_loss_val, steps_taken = model(
    dummy_images,
    max_steps=max_steps,
    threshold=threshold,
    labels=dummy_labels,
    criterion=criterion
)

print(f"Output Logits Shape: {logits.shape} (Expected: [4, 10])")
print(f"Steps Taken: {steps_taken} | CE Loss: {ce_loss_val:.4f} | Mean Aux Loss: {aux_loss_val:.4f}")

missing_grads = []
zero_grads = []
nan_grads = []

for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if param.grad is None:
        missing_grads.append(name)
    elif torch.isnan(param.grad).any():
        nan_grads.append(name)
    elif torch.norm(param.grad).item() == 0.0:
        zero_grads.append(name)

if not missing_grads and not zero_grads and not nan_grads:
    print("Sanity Check Passed: All parameters received valid, non-zero gradients!")
else:
    if missing_grads:
        print("Parameters with None grad:", missing_grads)
    if zero_grads:
        print("Parameters with 0.0 grad:", zero_grads)
    if nan_grads:
        print("Parameters with NaN grad:", nan_grads)

optimizer.zero_grad()

Output Logits Shape: torch.Size([4, 10]) (Expected: [4, 10])
Steps Taken: 12.0 | CE Loss: 2.3066 | Mean Aux Loss: 1.0025
Sanity Check Passed: All parameters received valid, non-zero gradients!


In [4]:
train_loader, test_loader = get_cifar10_dataloaders(
    data_dir="./data", batch_size=batch_size, num_workers=2, eval_noise_std=0.0
)

In [5]:
def train_one_epoch(model, loader, optimizer, criterion, max_steps, threshold):
    model.train()
    total_ce_loss = 0.0
    total_aux_loss = 0.0
    total_steps = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        # Forward pass also executes per-tick and final-tick backward() calls internally
        logits, ce_loss_val, aux_loss_val, steps_taken = model(
            images,
            max_steps=max_steps,
            threshold=threshold,
            labels=labels,
            criterion=criterion
        )

        # Clip accumulated multi-step gradients for stability, then update weights
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        batch_count = images.size(0)
        total_ce_loss += ce_loss_val * batch_count
        total_aux_loss += aux_loss_val * batch_count
        total_steps += steps_taken * batch_count

        preds = logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total += batch_count

    return (
        total_ce_loss / total,
        total_aux_loss / total,
        100.0 * correct / total,
        total_steps / total
    )


@torch.no_grad()
def evaluate(model, loader, max_steps, threshold):
    model.eval()
    correct = 0
    total = 0
    total_steps = 0.0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits, steps_taken = model(
            images, max_steps=max_steps, threshold=threshold
        )

        batch_count = images.size(0)
        preds = logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total += batch_count
        total_steps += steps_taken * batch_count

    return 100.0 * correct / total, total_steps / total

# Train the model

In [6]:
# Run Training Loop
os.makedirs("./checkpoints", exist_ok=True)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
best_val_acc = 0.0

for epoch in range(1, epochs + 1):
    start_time = time.time()

    train_ce, train_aux, train_acc, train_steps = train_one_epoch(
        model, train_loader, optimizer, criterion, max_steps, threshold
    )
    val_acc, val_steps = evaluate(model, test_loader, max_steps, threshold)
    scheduler.step()

    elapsed = time.time() - start_time
    print(
        f"Epoch {epoch:02d}/{epochs:02d} ({elapsed:.1f}s) | "
        f"Train CE: {train_ce:.4f} | Aux: {train_aux:.4f} | "
        f"Train Acc: {train_acc:.2f}% (Steps: {train_steps:.2f}) | "
        f"Val Acc: {val_acc:.2f}% (Steps: {val_steps:.2f})"
    )

    # Save checkpoint payload
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "train_ce": train_ce,
        "train_aux": train_aux,
        "train_acc": train_acc,
        "val_acc": val_acc,
        "val_steps": val_steps,
    }
    torch.save(checkpoint, "./checkpoints/flb_vit_latest.pt")
    torch.save(checkpoint, f"./checkpoints/flb_vit_epoch_{epoch:02d}.pt")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(checkpoint, "./checkpoints/flb_vit_best.pt")

Epoch 01/20 (189.6s) | Train CE: 1.6824 | Aux: 0.1158 | Train Acc: 37.86% (Steps: 12.00) | Val Acc: 49.79% (Steps: 11.98)
Epoch 02/20 (189.5s) | Train CE: 1.3181 | Aux: 0.0745 | Train Acc: 52.34% (Steps: 12.00) | Val Acc: 55.09% (Steps: 12.00)
Epoch 03/20 (185.3s) | Train CE: 1.1734 | Aux: 0.0495 | Train Acc: 57.75% (Steps: 12.00) | Val Acc: 59.90% (Steps: 12.00)
Epoch 04/20 (188.3s) | Train CE: 1.0882 | Aux: 0.0288 | Train Acc: 61.11% (Steps: 12.00) | Val Acc: 62.85% (Steps: 12.00)
Epoch 05/20 (186.0s) | Train CE: 1.0249 | Aux: 0.0157 | Train Acc: 63.38% (Steps: 12.00) | Val Acc: 64.76% (Steps: 12.00)
Epoch 06/20 (186.0s) | Train CE: 0.9778 | Aux: 0.0099 | Train Acc: 64.93% (Steps: 12.00) | Val Acc: 67.11% (Steps: 12.00)
Epoch 07/20 (186.5s) | Train CE: 0.9352 | Aux: 0.0083 | Train Acc: 66.77% (Steps: 12.00) | Val Acc: 67.53% (Steps: 12.00)
Epoch 08/20 (186.0s) | Train CE: 0.8977 | Aux: 0.0085 | Train Acc: 68.14% (Steps: 12.00) | Val Acc: 69.04% (Steps: 12.00)
Epoch 09/20 (186.9s) | T